[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/01_fundamentals/autograd.ipynb)



# 02 . Autograd

## Concept — Autograd (revision notes)

**What it is**
- Autograd is PyTorch's automatic differentiation engine.
- It records every operation performed on tensors with `requires_grad=True`,
  builds a graph of those operations, and computes gradients of any output
  w.r.t. any input by walking the graph backward — the chain rule, automated.

**Why it exists**
- Training a neural network means computing the gradient of a loss w.r.t.
  potentially millions of parameters.
- Doing this by hand is infeasible. Autograd computes it exactly, in one
  `.backward()` call.

**The 3 key attributes every tensor has**
- `.requires_grad` — should operations on this tensor be tracked? (`bool`)
- `.grad_fn` — the function that created this tensor. `None` for leaf
  tensors you created directly (e.g. model parameters, raw input data).
- `.grad` — where the computed gradient accumulates after `.backward()`.
  Only populated for **leaf** tensors with `requires_grad=True`, by default.

**How the graph works**
- Every op on a tracked tensor builds a **directed acyclic graph (DAG)** on
  the fly: leaves are your inputs/parameters, the root is your final output
  (usually the loss).
- Calling `.backward()` on the root walks this graph from output back to
  inputs, applying the chain rule at every node.
- The result is **accumulated** (added to, not overwritten) into each leaf's
  `.grad` — this is exactly why you must call `optimizer.zero_grad()` (or
  `tensor.grad.zero_()`) every training iteration, or gradients from
  previous steps pile up on top of new ones.

**Define-by-run**
- The graph is built dynamically, fresh, every forward pass, tracing your
  actual Python control flow as it executes.
- This is what "define-by-run" (as opposed to a static graph) means, and
  it's why you can freely use regular Python `if`/`for` loops inside a
  model's `forward()`.

**The graph is consumed once by default**
- After `.backward()`, the graph is freed to save memory.
- Calling `.backward()` a second time on the same graph without
  `retain_graph=True` raises a `RuntimeError` — a very common bug when
  `.backward()` gets accidentally called more than once per forward pass.

**Stopping gradient tracking**
- `torch.no_grad()` — a context manager; anything computed inside it has
  `requires_grad=False` results, regardless of the inputs. Use during
  evaluation/inference — saves memory and time.
- `.detach()` — returns a new tensor sharing the same data but with no
  autograd history. Use when you want a value without its computation graph
  (e.g. logging a loss, or feeding a value into a place that shouldn't
  backprop through it).
- Only floating-point tensors can `require_grad`. Integer tensors (labels,
  token ids) never track gradients — this is expected and correct.

**Non-scalar backward**
- `.backward()` requires a scalar output by default (like a loss value).
- For a non-scalar output, you must either reduce it first (`.sum()`,
  `.mean()`) or pass an explicit `gradient=` tensor of matching shape.

**Gradient accumulation as a technique**
- Because gradients accumulate by default, you can deliberately call
  `.backward()` several times across several mini-batches WITHOUT zeroing in
  between, then call `optimizer.step()` once — simulating a larger batch
  size than fits in memory. This is a legitimate production technique, not
  a bug, when done on purpose.


### 1. The most basic gradient: y = x^2, dy/dx = 2x

**What this cell does (plain language):** we create a single tensor that
tracks gradients, compute `y = x**2`, call `.backward()`, and print the
resulting gradient — comparing it against the value you'd get by hand from
calculus (`dy/dx = 2x`).


In [1]:
import torch

x = torch.tensor(3.0, requires_grad=True)
y = x ** 2
y.backward()
print(f"x = {x.item()}, y = {y.item()}, dy/dx = {x.grad.item()} (expected 2*3=6)")


x = 3.0, y = 9.0, dy/dx = 6.0 (expected 2*3=6)


### 2. Gradient accumulation — why you must zero_grad()

**What this cell does (plain language):** we call `.backward()` three times
in a row on the SAME tensor's gradient without resetting it in between, and
watch the gradient value grow each time instead of staying correct — then we
show the fix (`.grad.zero_()` every iteration), which is exactly what
`optimizer.zero_grad()` does for you in a real training loop.


In [2]:
x = torch.tensor(2.0, requires_grad=True)
print("WITHOUT zero_grad (grad keeps accumulating across steps):")
for step in range(3):
    y = x ** 2
    y.backward()  # dy/dx = 2x = 4 each time, but it ADDS to the existing grad
    print(f"  step {step}: x.grad = {x.grad.item()}")

x.grad.zero_()  # reset before demonstrating the correct pattern
print("\nWITH zero_grad each step (the correct pattern used in real training loops):")
for step in range(3):
    x.grad.zero_()
    y = x ** 2
    y.backward()
    print(f"  step {step}: x.grad = {x.grad.item()}  <- stays correct, doesn't pile up")


WITHOUT zero_grad (grad keeps accumulating across steps):
  step 0: x.grad = 4.0
  step 1: x.grad = 8.0
  step 2: x.grad = 12.0

WITH zero_grad each step (the correct pattern used in real training loops):
  step 0: x.grad = 4.0  <- stays correct, doesn't pile up
  step 1: x.grad = 4.0  <- stays correct, doesn't pile up
  step 2: x.grad = 4.0  <- stays correct, doesn't pile up


### 3. Leaf vs non-leaf tensors

**What this cell does (plain language):** we create a leaf tensor `a` and
derive a non-leaf tensor `b` from it (`b = a * 3`), then call `.backward()`
and check `.grad` on both — showing that only the leaf gets a populated
`.grad` by default, and how to opt in to keeping a non-leaf's gradient too.


In [3]:
a = torch.tensor(2.0, requires_grad=True)  # leaf: created directly
b = a * 3                                   # non-leaf: result of an operation
print("a.is_leaf:", a.is_leaf, "| b.is_leaf:", b.is_leaf)
print("a.grad_fn:", a.grad_fn)  # None - it's a leaf, nothing created it
print("b.grad_fn:", b.grad_fn)  # MulBackward - records that b came from a multiplication

b.backward()
print("\na.grad:", a.grad)  # populated - leaves get .grad by default
print("b.grad (not retained, expect None with a UserWarning):", b.grad)

# To keep a non-leaf's gradient too, call .retain_grad() on it BEFORE backward()
c = torch.tensor(2.0, requires_grad=True)
d = c * 3
d.retain_grad()  # opt in to keeping this intermediate gradient
d.backward()
print("\nd.grad after retain_grad():", d.grad)


a.is_leaf: True | b.is_leaf: False
a.grad_fn: None
b.grad_fn: <MulBackward0 object at 0x7f740f225e70>

a.grad: tensor(3.)
b.grad (not retained, expect None with a UserWarning): None

d.grad after retain_grad(): tensor(1.)


/tmp/ipykernel_2047/3282807275.py:9: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  print("b.grad (not retained, expect None with a UserWarning):", b.grad)


### 4. torch.no_grad() — for inference and evaluation

**What this cell does (plain language):** we do a computation inside a
`torch.no_grad()` block and show the result comes out with
`requires_grad=False`, even though the input tensor required gradients —
proving that no_grad genuinely turns off graph-building, not just hides it.


In [4]:
w = torch.tensor(5.0, requires_grad=True)

normal_result = w * 2
print("outside no_grad, result.requires_grad:", normal_result.requires_grad)

with torch.no_grad():
    result = w * 2
print("inside no_grad, result.requires_grad:", result.requires_grad)

# a very common real pattern: evaluating a model without building a graph
def predict(w, x):
    with torch.no_grad():
        return w * x

pred = predict(w, torch.tensor(10.0))
print("\nprediction (no graph attached):", pred, "| requires_grad:", pred.requires_grad)


outside no_grad, result.requires_grad: True
inside no_grad, result.requires_grad: False

prediction (no graph attached): tensor(50.) | requires_grad: False


### 5. .detach() — keep the value, drop the history

**What this cell does (plain language):** we take a tensor that requires
gradients, `.detach()` it, and show the detached version has no gradient
tracking while still holding the identical numeric value — useful for
logging or feeding a value somewhere it shouldn't backprop through.


In [5]:
loss = (w ** 2)
loss_value_only = loss.detach()

print("loss.requires_grad:", loss.requires_grad)
print("loss_value_only.requires_grad:", loss_value_only.requires_grad)
print("same numeric value:", loss.item() == loss_value_only.item())

# .detach() shares memory with the original - modifying it in place would
# affect the original tensor too (same "view" idea as with plain tensors)
print("\nshare memory:", loss.data_ptr() == loss_value_only.data_ptr())


loss.requires_grad: True
loss_value_only.requires_grad: False
same numeric value: True

share memory: True


### 6. Calling backward() twice — the classic error

**What this cell does (plain language):** we call `.backward()` on the same
computation graph twice in a row, without `retain_graph=True`, to show you
the exact error message this produces — then show the fix and why you'd
rarely actually need it in normal training code.


In [6]:
p = torch.tensor(1.0, requires_grad=True)
out = p ** 3
out.backward()
try:
    out.backward()  # graph was already freed after the first backward()
except RuntimeError as e:
    print("Error on second backward():", str(e)[:120], "...")

# Fix: retain_graph=True if you genuinely need to backward through the
# SAME graph twice (rare in normal training - usually a sign of a design issue,
# like accidentally reusing an intermediate result across two unrelated losses).
p2 = torch.tensor(1.0, requires_grad=True)
out2 = p2 ** 3
out2.backward(retain_graph=True)
out2.backward()  # works now, because we kept the graph alive
print("\nWith retain_graph=True, p2.grad after two backward calls:", p2.grad.item())
# note: it's now 3*1^2 + 3*1^2 = 6, because both calls ADDED to p2.grad


Error on second backward(): Trying to backward through the graph a second time (or directly access saved tensors after they have already been freed) ...

With retain_graph=True, p2.grad after two backward calls: 6.0


### 7. Gradient w.r.t. a vector — scalar requirement

**What this cell does (plain language):** we try calling `.backward()` on a
non-scalar (vector) output directly and show the error, then show the two
standard fixes: reducing to a scalar first, or passing an explicit gradient
tensor of the same shape.


In [7]:
v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out_vec = v * 2  # shape (3,), NOT a scalar
try:
    out_vec.backward()  # fails: backward() needs a scalar unless you specify gradient=
except RuntimeError as e:
    print("Error:", str(e)[:120], "...")

# Fix option 1: reduce to scalar first (this is what loss functions do internally)
out_vec = v * 2
out_vec.sum().backward()
print("\nv.grad after .sum().backward():", v.grad)

# Fix option 2: pass an explicit gradient tensor matching out_vec's shape
v.grad.zero_()
out_vec2 = v * 2
out_vec2.backward(torch.ones_like(out_vec2))
print("v.grad after explicit gradient arg:", v.grad)


Error: grad can be implicitly created only for scalar outputs ...

v.grad after .sum().backward(): tensor([2., 2., 2.])
v.grad after explicit gradient arg: tensor([2., 2., 2.])


### 8. Multi-variable gradients — two inputs, one output

**What this cell does (plain language):** we compute a function of TWO
variables at once, `z = x*y + y**2`, call `.backward()` once, and show both
`x.grad` and `y.grad` get populated correctly in that single call — you
don't need separate backward calls per input variable.


In [8]:
x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)

z = x * y + y ** 2
z.backward()

print("dz/dx (expected y = 3):", x.grad.item())
print("dz/dy (expected x + 2y = 2 + 6 = 8):", y.grad.item())

# ONE backward() call computes gradients for EVERY leaf tensor that
# contributed to the output, in a single backward pass through the graph.


dz/dx (expected y = 3): 3.0
dz/dy (expected x + 2y = 2 + 6 = 8): 8.0


### 9. Gradient accumulation as a deliberate technique

**What this cell does (plain language):** we simulate training with an
effective batch size of 16 while only ever holding 4 samples in memory at
once, by accumulating gradients across 4 mini-batches of size 4 before
calling `optimizer.step()` — a real production technique for training with
limited GPU memory.


In [9]:
import torch.nn as nn

model = nn.Linear(10, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

mini_batches = [torch.randn(4, 10) for _ in range(4)]  # 4 batches of 4 = effective batch of 16
targets = [torch.randn(4, 1) for _ in range(4)]

optimizer.zero_grad()  # zero ONCE, before the accumulation loop, not per mini-batch
for x_batch, y_batch in zip(mini_batches, targets):
    pred = model(x_batch)
    loss = ((pred - y_batch) ** 2).mean()
    loss = loss / len(mini_batches)  # scale down so the accumulated grad matches a true batch-of-16 average
    loss.backward()  # accumulates into .grad, does NOT overwrite

optimizer.step()  # one parameter update, based on gradients from all 4 mini-batches combined
print("Completed one accumulated update using 4 mini-batches of 4 (effective batch size 16)")
print("weight grad after accumulation (should be non-zero, combining all 4 batches):")
print(model.weight.grad)


Completed one accumulated update using 4 mini-batches of 4 (effective batch size 16)
weight grad after accumulation (should be non-zero, combining all 4 batches):
tensor([[ 0.0546, -0.3354,  0.0487,  0.0364, -1.4910, -0.0895, -0.6644, -0.4726,
         -1.2750, -0.1125]])


### 10. Inspecting the graph: grad_fn chains

**What this cell does (plain language):** we build a small chain of
operations and walk `.grad_fn` and its `.next_functions` to see the actual
structure of the autograd graph PyTorch built behind the scenes — useful for
genuinely understanding (or debugging) what "the graph" means concretely.


In [10]:
x = torch.tensor(2.0, requires_grad=True)
y = x * 3
z = y + 1
out = z ** 2

print("out.grad_fn:", out.grad_fn)                       # PowBackward
print("out.grad_fn.next_functions:", out.grad_fn.next_functions)  # points back to z's AddBackward

# walk the chain manually
node = out.grad_fn
depth = 0
while node is not None:
    print("  " * depth + str(node))
    next_fns = node.next_functions
    node = next_fns[0][0] if next_fns and next_fns[0][0] is not None else None
    depth += 1


out.grad_fn: <PowBackward0 object at 0x7f74242a8520>
out.grad_fn.next_functions: ((<AddBackward0 object at 0x7f73f06d9120>, 0),)


## Common bugs

- **"My gradients are None"**
  Usually one of: the tensor doesn't have `requires_grad=True`; you're
  checking `.grad` on a non-leaf tensor without calling `.retain_grad()`
  first; it was detached somewhere upstream; or the computation happened
  inside `torch.no_grad()` by accident. Diagnose by printing
  `tensor.requires_grad` and `tensor.is_leaf` right before you expect a
  gradient to exist.

- **"Trying to backward through the graph a second time"**
  Called `.backward()` more than once sharing part of the same graph,
  without `retain_graph=True`. Usually a sign of a loop/logic bug (like
  calling `.backward()` twice per iteration by accident) rather than a
  genuine need for `retain_graph=True`.

- **"grad can be implicitly created only for scalar outputs"**
  Called `.backward()` on a tensor with more than one element without
  specifying what gradient to backprop. Fix: reduce to a scalar first
  (`.sum()`, `.mean()`) — this is exactly what loss functions do — or pass
  an explicit `gradient=` tensor matching the output's shape.

- **Loss is `nan`/`inf` after a few steps (exploding gradients)**
  Learning rate too high, missing gradient clipping, bad initialization, or
  a numerically unstable op (`log(0)`, division by a value that can hit
  zero). Diagnose by printing per-layer gradient norms; fix with
  `torch.nn.utils.clip_grad_norm_`, a lower learning rate, or fixing the
  unstable op.

- **Loss never decreases (vanishing gradients or frozen params)**
  Could be a genuinely deep-network gradient issue, or you accidentally set
  `requires_grad=False` on parameters that should be learnable (common
  after loading a pretrained model and forgetting to unfreeze the layers you
  meant to fine-tune). Diagnose with
  `for name, p in model.named_parameters(): print(name, p.requires_grad, p.grad is None)`.

- **In-place op breaks the backward pass**
  `RuntimeError: one of the variables needed for gradient computation has
  been modified by an inplace operation`. Caused by an in-place op
  (`x.add_(1)`, `x += y`) on a tensor autograd needed to keep unmodified for
  the backward pass. Fix: use the non-in-place version in the forward path
  of anything being trained.

- **Memory grows every iteration of the training loop**
  You're accumulating tensors that still carry their autograd graph, e.g.
  `losses.append(loss)` instead of `losses.append(loss.item())`. Every
  appended `loss` keeps its ENTIRE computation graph alive in memory. Fix:
  always `.item()` or `.detach()` values you're only logging, not continuing
  to backprop through.

- **Forgot to scale loss when accumulating gradients over mini-batches**
  If you split a batch into N mini-batches and call `.backward()` on each
  without dividing the loss by N, the accumulated gradient ends up N times
  larger than intended, effectively acting like a much higher learning rate.
  Fix: divide the loss by the number of accumulation steps before calling
  `.backward()`, as shown in section 9.


## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1 - Compute by hand, then verify.**
For `f(x) = 3x^3 + 2x^2 - 5`, what is `df/dx` at `x = 2`? Compute it with pen
and paper first, then verify with autograd.

In [11]:
# Your attempt for Exercise 1 here\n

**Solution 1**

In [12]:
# Solution 1
# By hand: f(x) = 3x^3 + 2x^2 - 5
# f'(x) = 9x^2 + 4x
# at x=2: 9*(4) + 4*(2) = 36 + 8 = 44

x = torch.tensor(2.0, requires_grad=True)
f = 3 * x**3 + 2 * x**2 - 5
f.backward()
print("df/dx at x=2:", x.grad.item(), "(expected 44)")


df/dx at x=2: 44.0 (expected 44)


**Exercise 2 - Why None?**
Explain why `x.grad` is `None` in this snippet, then fix it so it isn't:
```python
x = torch.tensor([1.0, 2.0, 3.0])
y = (x ** 2).sum()
y.backward()
print(x.grad)
```

In [ ]:
# Your attempt for Exercise 2 here\n

**Solution 2**

In [14]:
# Solution 2
# Why None: `x` was created WITHOUT requires_grad=True, so autograd never
# tracked any operations on it, and there's no graph to backward through
# for x specifically - PyTorch doesn't even build the tracking machinery.

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)  # the fix
y = (x ** 2).sum()
y.backward()
print("x.grad:", x.grad)  # now populated: d(sum(x^2))/dx = 2x = [2, 4, 6]


x.grad: tensor([2., 4., 6.])


**Exercise 3 - Multi-variable gradient by hand.**
For `z = x*y + y**2` at `x=2, y=3`, compute `dz/dx` and `dz/dy` by hand, then
verify with autograd. Do you need one `.backward()` call or two to get both?

In [15]:
# Your attempt for Exercise 3 here\n

**Solution 3**

In [16]:
# Solution 3
# By hand: z = x*y + y^2
# dz/dx = y = 3
# dz/dy = x + 2y = 2 + 6 = 8

x = torch.tensor(2.0, requires_grad=True)
y = torch.tensor(3.0, requires_grad=True)
z = x * y + y ** 2
z.backward()  # ONE call computes gradients for BOTH x and y simultaneously
print("dz/dx:", x.grad.item(), "(expected 3)")
print("dz/dy:", y.grad.item(), "(expected 8)")


dz/dx: 3.0 (expected 3)
dz/dy: 8.0 (expected 8)


**Exercise 4 - Freeze half a model.**
Given a two-layer `nn.Module` (`layer1`, `layer2`), write code that freezes
`layer1`'s parameters (no gradient updates) while `layer2` still trains
normally. Verify by checking `.grad` on both layers' parameters after a
backward pass.

In [17]:
# Your attempt for Exercise 4 here\n

**Solution 4**

In [18]:
# Solution 4
import torch.nn as nn

class TwoLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(10, 10)
        self.layer2 = nn.Linear(10, 1)
    def forward(self, x):
        return self.layer2(torch.relu(self.layer1(x)))

model = TwoLayer()
for p in model.layer1.parameters():
    p.requires_grad = False

x = torch.randn(4, 10)
target = torch.randn(4, 1)
loss = ((model(x) - target) ** 2).mean()
loss.backward()

print("layer1.weight.grad is None (frozen, correct):", model.layer1.weight.grad is None)
print("layer2.weight.grad is None (should train, correct):", model.layer2.weight.grad is None)


layer1.weight.grad is None (frozen, correct): True
layer2.weight.grad is None (should train, correct): False


**Exercise 5 - Spot and fix the in-place bug.**
This raises a runtime error during `.backward()` - identify why and fix it:
```python
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * 2
y += 1  # <- problem is here
loss = y.sum()
loss.backward()
```

In [19]:
# Your attempt for Exercise 5 here\n

**Solution 5**

In [20]:
# Solution 5
# Bug: `y += 1` is an in-place op on y, which is needed unmodified for the
# backward pass since it was created from x (autograd needs y's pre-add
# value, or at least the computational relationship, to compute dy/dx
# correctly through the chain). The += modifies y's data directly.
# Fix: use the non-in-place version, y = y + 1, which creates a NEW tensor
# instead of overwriting y.

x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x * 2
y = y + 1  # the fix: non-in-place, creates a new tensor
loss = y.sum()
loss.backward()
print("x.grad:", x.grad)  # works fine now


x.grad: tensor([2., 2.])


**Exercise 6 - Gradient accumulation for a target effective batch size.**
Write a training loop that simulates an effective batch size of 32 using a
physical mini-batch size of 8 (so 4 accumulation steps), correctly scaling
the loss and calling `optimizer.step()` only once per 4 mini-batches.

In [21]:
# Your attempt for Exercise 6 here\n

**Solution 6**

In [22]:
# Solution 6
import torch.nn as nn

model = nn.Linear(10, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

physical_batch_size = 8
accumulation_steps = 4  # 8 * 4 = effective batch size of 32

optimizer.zero_grad()
for step in range(accumulation_steps):
    x = torch.randn(physical_batch_size, 10)
    y = torch.randn(physical_batch_size, 1)
    pred = model(x)
    loss = ((pred - y) ** 2).mean()
    loss = loss / accumulation_steps  # scale so the accumulated gradient matches a true batch of 32
    loss.backward()

optimizer.step()  # ONE update using gradients accumulated over all 4 mini-batches
print("Completed one update simulating effective batch size 32 from 4 mini-batches of 8")


Completed one update simulating effective batch size 32 from 4 mini-batches of 8


**Exercise 7 - retain_grad() on an intermediate value.**
Given `a = torch.tensor(3.0, requires_grad=True)` and `b = a ** 2`,
`c = b * 2`, write code to inspect `b.grad` after calling `c.backward()` -
first showing it's `None`, then fixing it with `retain_grad()`.

In [23]:
# Your attempt for Exercise 7 here\n

**Solution 7**

In [24]:
# Solution 7
a = torch.tensor(3.0, requires_grad=True)
b = a ** 2
c = b * 2

c.backward()
print("b.grad WITHOUT retain_grad (expect None):", b.grad)

# redo with retain_grad
a2 = torch.tensor(3.0, requires_grad=True)
b2 = a2 ** 2
b2.retain_grad()  # must call BEFORE backward()
c2 = b2 * 2
c2.backward()
print("b2.grad WITH retain_grad:", b2.grad)  # dc/db = 2


b.grad WITHOUT retain_grad (expect None): None
b2.grad WITH retain_grad: tensor(2.)


/tmp/ipykernel_2047/1208032262.py:7: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:494.)
  print("b.grad WITHOUT retain_grad (expect None):", b.grad)


**Exercise 8 - Detach mid-computation.**
Write a function that computes `y = f(x)` where part of the computation
should NOT be backpropagated through (e.g. a target value derived from `x`
that should be treated as a constant). Use `.detach()` correctly to achieve this.

In [25]:
# Your attempt for Exercise 8 here\n

**Solution 8**

In [26]:
# Solution 8
def compute_with_constant_target(x):
    # imagine `target` should be treated as a fixed constant derived from x,
    # NOT something we backprop through (a common pattern in self-supervised
    # or target-network style setups)
    target = (x ** 2).detach()  # detach breaks the graph connection here
    prediction = x * 3
    loss = ((prediction - target) ** 2).mean()
    return loss

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
loss = compute_with_constant_target(x)
loss.backward()
print("x.grad:", x.grad)
print("gradient only flows through `prediction = x*3`, not through the detached target")


x.grad: tensor([4., 4., 0.])
gradient only flows through `prediction = x*3`, not through the detached target


**Exercise 9 - Vector backward with a custom gradient.**
Given `v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)` and
`out = v ** 2`, use the explicit `gradient=` argument to `.backward()` to
compute a WEIGHTED sum of gradients, where the weights are `[1.0, 0.0, 2.0]`
(i.e. you want `1*d(out[0]) + 0*d(out[1]) + 2*d(out[2])`). Verify the result
against computing it by hand.

In [27]:
# Your attempt for Exercise 9 here\n

**Solution 9**

In [28]:
# Solution 9
v = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
out = v ** 2

weights = torch.tensor([1.0, 0.0, 2.0])
out.backward(gradient=weights)

# By hand: d(out_i)/d(v_i) = 2*v_i, weighted per-element = weight_i * 2 * v_i
# = [1*2*1, 0*2*2, 2*2*3] = [2, 0, 12]
print("v.grad:", v.grad, "(expected [2, 0, 12])")


v.grad: tensor([ 2.,  0., 12.]) (expected [2, 0, 12])


**Exercise 10 - Diagnose a frozen-model bug.**
You loaded a pretrained model, and after training, none of the weights
changed. Write code that would help you diagnose whether this is because
(a) `requires_grad=False` was accidentally set everywhere, or (b) the
optimizer was accidentally constructed with a DIFFERENT model's parameters
(a realistic copy-paste mistake). Simulate both bugs and show how your
diagnostic code catches each one.

In [29]:
# Your attempt for Exercise 10 here\n

**Solution 10**

In [30]:
# Solution 10
import torch.nn as nn

# Bug (a): requires_grad=False set everywhere by accident
model_a = nn.Linear(5, 1)
for p in model_a.parameters():
    p.requires_grad = False
optimizer_a = torch.optim.SGD(model_a.parameters(), lr=0.1)

x = torch.randn(3, 5)
loss_a = model_a(x).sum()

# With EVERYTHING frozen, the output has no grad_fn at all, so .backward()
# itself raises an error - that failure IS the diagnostic signal for bug (a).
try:
    loss_a.backward()
    print("Bug (a): backward() succeeded (unexpected)")
except RuntimeError as e:
    print("Bug (a) diagnostic - backward() failed immediately:", str(e)[:80])
    print("Bug (a) diagnostic - any trainable params?",
          any(p.requires_grad for p in model_a.parameters()), "<- False confirms the bug")

# Bug (b): optimizer built with the WRONG model's parameters (a realistic
# copy-paste mistake - e.g. optimizer accidentally built from an old/unused
# model instance instead of the one you're actually training)
model_b = nn.Linear(5, 1)
decoy_model = nn.Linear(5, 1)  # a completely different, unrelated model
optimizer_b = torch.optim.SGD(decoy_model.parameters(), lr=0.1)  # oops - wrong model!

loss_b = model_b(x).sum()
loss_b.backward()  # succeeds fine - model_b's params require grad and have a grad_fn
print("\nBug (b) diagnostic - model_b params have grad:", model_b.weight.grad is not None)

# the diagnostic: check whether the optimizer's tracked parameters are
# actually the SAME objects as the model you just computed gradients for
optimizer_params = {id(p) for group in optimizer_b.param_groups for p in group['params']}
model_b_params = {id(p) for p in model_b.parameters()}
print("Bug (b) diagnostic - optimizer tracks any of model_b's actual parameters:",
      len(optimizer_params & model_b_params) > 0, "<- False confirms the bug")
print("-> gradients exist on model_b, but optimizer.step() would update decoy_model instead, not model_b")


Bug (a) diagnostic - backward() failed immediately: element 0 of tensors does not require grad and does not have a grad_fn
Bug (a) diagnostic - any trainable params? False <- False confirms the bug

Bug (b) diagnostic - model_b params have grad: True
Bug (b) diagnostic - optimizer tracks any of model_b's actual parameters: False <- False confirms the bug
-> gradients exist on model_b, but optimizer.step() would update decoy_model instead, not model_b
